### Packages

In [1]:
import pickle
import numpy as np
import math
import matplotlib.pyplot as plt
import tarfile
import urllib.request
import os
import shutil
from torch_gradient_computations import ComputeGradsWithTorch



In [2]:
import torch


In [3]:

FILEPATH="Datasets/cifar-10-batches-py/" # we add the directory path first 


### Read the file, this will be used to access the train, val and test batches.

In [4]:

def LoadBatch(filename):
    with open(filename, "rb") as fo:
        batch = pickle.load(fo, encoding="bytes")

    X = batch[b"data"].astype(np.float64).T / 255.0   # 3072 x 10000  divide by 255.0 so to keep everything small 
    y = np.array(batch[b"labels"])                    # length 10000, since we have 1000 images

    K = 10
    n = X.shape[1]
    Y = np.zeros((K, n), dtype=X.dtype)
    Y[y, np.arange(n)] = 1 # each column represent the class representation of one image 

    return X, Y, y

 ### Normalize the data: X_train, X_validation, X_test

In [5]:
def normalize(X, mean_X, std_X):
    return (X - mean_X) / std_X

### Access the training, validation and test data.

In [6]:


X_train,Y_train,y_train=LoadBatch(FILEPATH+"data_batch_1")

X_validation,Y_validation,y_validation=LoadBatch(FILEPATH+"data_batch_2")

X_test,Y_test,y_test=LoadBatch(FILEPATH+"test_batch")

d = X_train.shape[0]

mean_X = np.mean(X_train, axis=1).reshape(d, 1)
std_X  = np.std(X_train, axis=1).reshape(d, 1)

X_train = normalize(X_train, mean_X, std_X)
X_validation = normalize(X_validation, mean_X, std_X)
X_test = normalize(X_test, mean_X, std_X)

### Initialize parameters $W$ and $b$

In [7]:
def init_network(X_train):

    K = 10
    d = X_train.shape[0]

    rng = np.random.default_rng()
    BitGen = type(rng.bit_generator)
    seed = 42
    rng.bit_generator.state = BitGen(seed).state

    init_net = {
        'W': 0.01 * rng.standard_normal((K, d)),
        'b': np.zeros((K, 1))
    }
    return init_net



### Softmax function

In [8]:
def softmax(s):
    s = s - np.max(s, axis=0, keepdims=True)   # stability
    exp_s = np.exp(s)
    return exp_s / np.sum(exp_s, axis=0, keepdims=True)


### Define a function that applies:

\begin{equation*}
\begin{aligned}
\mathbf{s} & =W \mathbf{x}+\mathbf{b} \\
\mathbf{p} & =\operatorname{SOFTMAX}(\mathbf{s})
\end{aligned}
\end{equation*}

In [9]:
def ApplyNetwork(X,network):


    W=network['W']
    b=network['b']

    s=np.dot(W,X)
    P=softmax(s)
    return P

In [10]:
network=init_network(X_train)
P = ApplyNetwork(X_train[:, 0:100], network)
print(P.shape)
print(np.sum(P[:, 0]))

(10, 100)
1.0


### Compute

\begin{equation*}
J(\mathcal{D}, \lambda, W, b)=\frac{1}{|\mathcal{D}|} \sum_{(\mathbf{x}, y) \in \mathcal{D}} -\log \left(p_y\right)
\end{equation*}



In [11]:
def ComputeLoss(P, y):
    n = P.shape[1]
    correct_class_probs = P[y, np.arange(n)]
    L = -np.mean(np.log(correct_class_probs))
    return L



Now let us connect this to $W$ and $b$. Since

\begin{equation*}
s=W x+b
\end{equation*}

we use the chain rule.
Because $s_k=w_k^T x+b_k$, we get

\begin{equation*}
\frac{\partial s_k}{\partial w_k}=x, \quad \frac{\partial s_k}{\partial b_k}=1
\end{equation*}

and therefore

\begin{equation*}
\begin{gathered}
\frac{\partial l}{\partial W}=\frac{\partial l}{\partial s} \frac{\partial s}{\partial W}=(p-y) x^T \\
\frac{\partial l}{\partial b}=\frac{\partial l}{\partial s}=p-y
\end{gathered}
\end{equation*}


So the logic is:

\begin{equation*}
\ell \rightarrow \frac{\partial \ell}{\partial s}=p-y
\end{equation*}

\begin{equation*}
\begin{gathered}
\frac{\partial J}{\partial W}=\frac{1}{n} G X^T+2 \lambda W \\
\frac{\partial J}{\partial b}=\frac{1}{n} \sum_{i=1}^n g^{(i)}
\end{gathered}
\end{equation*}


### Here we calculate the gradient of $W$ and $b$

In [12]:

def BackwardPass(X, Y, P, network, lam):
    W = network['W']
    n = X.shape[1]

    G = P - Y

    grads = {}
    grads['W'] = (G @ X.T) / n + 2 * lam * W
    grads['b'] = np.sum(G, axis=1, keepdims=True) / n

    return grads

In [13]:

rng = np.random.default_rng()
BitGen = type(rng.bit_generator)
seed = 42
rng.bit_generator.state = BitGen(seed).state

d_small = 10
n_small = 3
lam = 0.0

small_net = {}
small_net['W'] = 0.01 * rng.standard_normal(size=(10, d_small))
small_net['b'] = np.zeros((10, 1))

X_small = X_train[0:d_small, 0:n_small]
Y_small = Y_train[:, 0:n_small]
y_small = y_train[0:n_small]

P_small = ApplyNetwork(X_small, small_net)

my_grads = BackwardPass(X_small, Y_small, P_small, small_net, lam)
torch_grads = ComputeGradsWithTorch(X_small, y_small, small_net)

print("max abs diff W:", np.max(np.abs(my_grads['W'] - torch_grads['W'])))
print("max abs diff b:", np.max(np.abs(my_grads['b'] - torch_grads['b'])))

def relative_error(a, b, eps=1e-10):
    return np.abs(a - b) / np.maximum(eps, np.abs(a) + np.abs(b))

rel_W = relative_error(my_grads['W'], torch_grads['W'])
rel_b = relative_error(my_grads['b'], torch_grads['b'])

print("max relative error W:", np.max(rel_W))
print("max relative error b:", np.max(rel_b))

max abs diff W: 1.1102230246251565e-16
max abs diff b: 5.551115123125783e-17
max relative error W: 2.1004151634013915e-15
max relative error b: 1.190200411005617e-16


In [19]:
lam = 0.1

P_small = ApplyNetwork(X_small, small_net)
my_grads = BackwardPass(X_small, Y_small, P_small, small_net, lam)
torch_grads = ComputeGradsWithTorch(X_small, y_small, small_net, lam)

print("max abs diff W:", np.max(np.abs(my_grads['W'] - torch_grads['W'])))
print("max abs diff b:", np.max(np.abs(my_grads['b'] - torch_grads['b'])))
print("max relative error W:", np.max(relative_error(my_grads['W'], torch_grads['W'])))
print("max relative error b:", np.max(relative_error(my_grads['b'], torch_grads['b'])))

max abs diff W: 1.1102230246251565e-16
max abs diff b: 5.551115123125783e-17
max relative error W: 4.5829256268066545e-15
max relative error b: 1.190200411005617e-16
